In [1]:
! pip install pandas requests python-dotenv
import pandas as pd
import requests
import io
import zipfile
import os

def avvia_estrazione_anac(config):
    output_file = config['nome_file_output']
    scrivi_header = not os.path.exists(output_file)

    print(f"--- Inizio estrazione periodo: {config['anno_inizio']} - {config['anno_fine']} ---")

    for anno in range(config['anno_inizio'], config['anno_fine'] + 1):
        for mese in range(1, 13):
            mese_str = f"{mese:02d}"
            url = f"https://dati.anticorruzione.it/opendata/download/dataset/cig-{anno}/filesystem/cig_csv_{anno}_{mese_str}.zip"
            
            try:
                print(f"\nTentativo: {anno}/{mese_str}...", end="\r")
                r = requests.get(url, timeout=30)
                r.raise_for_status()

                with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                    csv_nome = z.namelist()[0]
                    with z.open(csv_nome) as f:
                        # --- DIAGNOSI COLONNE ---
                        header_file = pd.read_csv(f, sep=';', encoding='latin1', nrows=0).columns.tolist()
                        
                        mancanti = [c for c in config['colonne_interesse'] if c not in header_file]
                        
                        if mancanti:
                            print(f"\n[ERRORE COLONNE] {anno}/{mese_str} saltato.")
                            print(f"Mancano: {mancanti}")
                            print(f"Colonne disponibili nel file ANAC: {header_file[:10]}... (e altre {len(header_file)-10})")
                            continue 

                        f.seek(0) 
                        reader = pd.read_csv(f, sep=';', encoding='latin1', usecols=config['colonne_interesse'], chunksize=40000)

                        for chunk in reader:
                            mask = True
                            for col, val in config['filtri_valore'].items():
                                mask &= (chunk[col].astype(str).str.upper() == str(val).upper())
                            
                            df_filtrato = chunk[mask].copy()

                            if not df_filtrato.empty:
                                if config['pulisci_importi'] and config['colonna_importo'] in df_filtrato.columns:
                                    c = config['colonna_importo']
                                    df_filtrato[c] = pd.to_numeric(df_filtrato[c].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False), errors='coerce')

                                df_filtrato.to_csv(output_file, mode='a', index=False, header=scrivi_header, sep=',')
                                scrivi_header = False
                                print(f"Scritte {len(df_filtrato)} righe per {anno}/{mese_str}!")

            except Exception:
                continue

    print(f"\n--- Processo completato. Dati salvati in: {os.path.abspath(output_file)} ---")

# ============================================================
# CONFIGURAZIONE (Placeholder da compilare dopo la diagnosi)
# ============================================================

configurazione_utente = {
    'anno_inizio': 2020, # < --- SCEGLI L'ANNO DI PARTENZA
    'anno_fine': 2023, # < --- SCEGLI L'ANNO DI FINE
    'colonne_interesse': ['cig', 'tipo_scelta_contraente', 'importo_lotto'], # <--- SCEGLI LE COLONNE CHE TI INTERESSANO
    'filtri_valore': {'tipo_scelta_contraente': 'PROCEDURA APERTA'},        # <--- SCEGLI I FILTRI CHE VUOI
    'nome_file_output': 'estrazione_anac_2020-2024.csv', # DAI UN NOME AL FILE
    'pulisci_importi': True,
    'colonna_importo': 'importo_lotto'
}

# ESECUZIONE
avvia_estrazione_anac(configurazione_utente)

--- Inizio estrazione periodo: 2020 - 2023 ---

Scritte 3958 righe per 2020/01!

Scritte 4709 righe per 2020/02!

Scritte 3909 righe per 2020/03!

Scritte 3064 righe per 2020/04!

Scritte 4381 righe per 2020/05!

Scritte 5547 righe per 2020/06!

Scritte 5240 righe per 2020/07!

Scritte 3214 righe per 2020/08!

Scritte 4039 righe per 2020/09!

Scritte 5206 righe per 2020/10!

Scritte 5055 righe per 2020/11!

Scritte 5432 righe per 2020/12!
Scritte 535 righe per 2020/12!

Scritte 4955 righe per 2021/01!

Scritte 3691 righe per 2021/02!

Scritte 5758 righe per 2021/03!

Scritte 5000 righe per 2021/04!
Scritte 246 righe per 2021/04!

Scritte 5052 righe per 2021/05!

Scritte 4742 righe per 2021/06!

Scritte 5067 righe per 2021/07!

Scritte 3226 righe per 2021/08!

Scritte 4921 righe per 2021/09!

Scritte 4406 righe per 2021/10!

Scritte 5182 righe per 2021/11!

Scritte 5061 righe per 2021/12!
Scritte 707 righe per 2021/12!

Scritte 6794 righe per 2022/01!

Scritte 3769 righe per 2022/02!

S